In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def reduce_mem_usage(df):

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    return df

In [ ]:
train_df = pd.read_csv('/kaggle/input/pubg-finish-placement-prediction/train_V2.csv')
test_df = pd.read_csv('/kaggle/input/pubg-finish-placement-prediction/test_V2.csv')
train = reduce_mem_usage(train_df)
test = reduce_mem_usage(test_df)

In [ ]:
from sklearn.preprocessing import LabelEncoder
def featuring(data,is_train):
    train = data
    
    
    train.loc[(train['matchType']=='squad-fpp') | (train['matchType']=='squad') | (train['matchType']=='normal-squad-fpp') |
                        (train['matchType']=='flarefpp') | (train['matchType']=='flaretpp') | (train['matchType']=='normal-squad'), 'matchType'] = 0
    
    train.loc[(train['matchType']=='duo') | (train['matchType']=='duo-fpp') | (train['matchType']=='normal-duo-fpp') |
                        (train['matchType']=='crashfpp') | (train['matchType']=='crashtpp')|(train['matchType']=='normal-duo'),'matchType'] = 1
    
    train.loc[(train['matchType']=='solo-fpp') | (train['matchType']=='solo') | (train['matchType']=='normal-solo-fpp') |
         (train['matchType']=='normal-solo'),'matchType'] = 2
    
   
    train['numPlayers'] = train.groupby('matchId')['Id'].transform('count')
    train['teamPlayers'] = train.groupId.map(train.groupId.value_counts())
    train['enemyPlayers'] = train['numPlayers'] - train['teamPlayers']

    m = train.groupby('matchId')
    train['killsPerc'] = m['kills'].rank(pct=True).values
    train['killPlacePerc'] = m['killPlace'].rank(pct=True).values
    train['walkDistancePerc'] = m['walkDistance'].rank(pct=True).values
    
    train['walkPerc_killsPerc'] = train['walkDistancePerc'] / train['killsPerc']
    
    train = train.fillna(0)
    train['kills_walkPerc'] = train['kills'] / train['walkDistancePerc'] # last
    train['walk_killsPerc'] = train['walkDistance'] / train['killsPerc'] # last
    
    train['Distance'] = train['rideDistance']+train['walkDistance']+train['swimDistance'] 
    
    train = train.fillna(0)
    #train['Distance_match'] = train['Distance'] / train['matchDuration'] # last
    
    train['walkoverheals'] = train['walkDistance'] / train['heals'] 
    train['walkoverkills'] = train['walkDistance'] / train['kills'] 
    train['teamwork'] = train['assists'] + train['revives'] 
    train['killStreakOverKills'] = train['killStreaks'] / train['kills']
    train['skills'] = train['headshotKills'] + train['roadKills']
    train['distanceOverWeapons'] = train['Distance'] / train['weaponsAcquired']
    
    train = train.fillna(0)
    
    train['buff'] = train['boosts'] + train['heals']
    train['headshotPerc'] = train['headshotKills'] / train['kills']
    train['killPlaceOverMaxPlace'] = train['killPlace'] / train['maxPlace']
    train['killOverWalkDistance'] = train['kills'] / train['walkDistance']
    
    train['totalDamage'] = train['kills'] + train['assists']
    train['buffperkill'] = train['buff'] / train['totalDamage']
    train['killPlacePerc'] = train['killPlace'] / train['numPlayers']
    
    train = train.fillna(0)
    
    
    #train.drop(['boosts','heals','killStreaks','DBNOs'],axis=1,inplace=True)
    train.drop(['headshotKills','roadKills','vehicleDestroys','rideDistance','swimDistance','matchDuration','rankPoints','killPoints','winPoints'],axis=1,inplace=True)
    
  
    
    if is_train: col = train.drop(['matchId','groupId','Id','matchType','numGroups','numPlayers','winPlacePerc','teamPlayers','enemyPlayers'],axis=1).columns
    else : col = train.drop(['matchId','groupId','Id','matchType','numGroups','numPlayers','teamPlayers','enemyPlayers'],axis=1).columns
        
    avg = train.groupby(['matchId','groupId'])[col].mean()
    avg_r = avg.groupby('matchId')[col].rank(pct=True).reset_index()
    
    avg = avg.reset_index()
    avg = avg.merge(avg_r, suffixes=['_mean','_mean_rank'], how='left', on=['matchId','groupId'])
    
    #train = train.merge(avg, how='left', on=['matchId','groupId'])
    
    ma = train.groupby(['matchId','groupId'])[col].max()
    ma_r = ma.groupby('matchId')[col].rank(pct=True).reset_index()
    
    ma = ma.reset_index()
    ma = ma.merge(ma_r, suffixes=['_max','_max_rank'], how='left', on=['matchId','groupId'])
    
    mi = train.groupby(['matchId','groupId'])[col].min()
    mi_r = mi.groupby('matchId')[col].rank(pct=True).reset_index()
    
    mi = mi.reset_index()
    mi = mi.merge(mi_r, suffixes=['_min','_min_rank'], how='left', on=['matchId','groupId'])
    
    mamin = train.groupby(['matchId'])[col].mean().reset_index()
    sizeg = train.groupby(['matchId','groupId']).size().reset_index(name='group_size')
    sizem = train.groupby(['matchId']).size().reset_index(name='match_size')
    
    train = train.merge(avg,how='left', on=['matchId','groupId'])
    train = train.merge(ma, how='left', on=['matchId','groupId'])
    train = train.merge(mi, how='left', on=['matchId','groupId'])
    train = train.merge(mamin, how='left',on=['matchId'], suffixes = ['','_match_mean'])
    train = train.merge(sizeg, how='left', on=['matchId','groupId'])
    train = train.merge(sizem, how='left',on=['matchId'])
    
    train = train.fillna(0)
    
    return train

In [ ]:
train1 = train.copy()#outlier_processing(train)
test1 = test.copy()#outlier_processing(test)

In [ ]:
#train1 = reduce_mem_usage(train1)
#test1 = reduce_mem_usage(test1)

In [ ]:
print('start featuring train')
train2 = featuring(train1,True)
print('start featuring test')
test2 = featuring(test1,False)
print('finish featuring')

In [ ]:
train2.dtypes.unique()

In [ ]:
train2 = reduce_mem_usage(train2)
test2 = reduce_mem_usage(test2)
print('finish reduce mem')

In [ ]:
def post_processing(data,pred):
    
    df = data[['Id','matchId','groupId','maxPlace','numGroups']]
    
    df['win_pred'] = pred
    
    #df = df[['Id','matchId','groupId','maxPlace','numGroups','win_pred']]
    
    df_g = df.groupby(['matchId','groupId']).first().reset_index()
    df_g['team_place'] = df_g.groupby(['matchId'])['win_pred'].rank()
    
    df_g['win_perc'] = (df_g['team_place'] - 1) / (df_g['numGroups'] - 1)
    df = df.merge(df_g[['win_perc','matchId','groupId']], on=['matchId','groupId'], how='left')
    
    df.loc[df['maxPlace']==0, 'win_perc'] = 0
    df.loc[df['maxPlace']==1, 'win_perc'] = 1
    df.loc[(df['maxPlace'] > 1) & (df['numGroups'] ==1), 'win_perc'] = 0
    df.loc[df['win_perc']<0, 'win_perc'] = 0
    df.loc[df['win_perc']>1, 'win_perc'] = 1
    df['win_perc'].fillna(df['win_pred'],inplace=True)
    
    subset = df.loc[df['maxPlace'] > 1]
    gap = 1 / (subset['maxPlace'].values-1)
    new_perc = np.around(subset['win_perc'].values / gap) * gap
    df.loc[df.maxPlace > 1, 'win_perc'] = new_perc
    
    df['winPlacePerc'] = df['win_perc']
    
    return df[['Id','winPlacePerc']]

In [ ]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

def lgb_train(tr,te):
    train = tr
    y = train['winPlacePerc']
    X = train.drop(['Id','winPlacePerc','matchType','groupId','matchId','maxPlace'], axis = 1)
    
    X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2, random_state=42)
    
    lightgbm = LGBMRegressor(random_state=42,n_estimators=20000,learning_rate = 0.05,early_stopping_round=100)
    lightgbm.fit(X_train,y_train,eval_set=[(X_val,y_val)], eval_metric='l1', verbose=1000)
    pred = lightgbm.predict(X_val)
    print(mean_absolute_error(y_val, pred))
    
    test = te
    predic = lightgbm.predict(test.drop(['Id','matchType','groupId','matchId','maxPlace'],axis=1))
    test['winPlacePerc'] = predic
    
    
    return test[['Id','winPlacePerc']]

In [ ]:
#test_all = lgb_train(train2,test2)
#test_all

In [ ]:
#test_pro = post_processing(test2,test_all['winPlacePerc'])
#test_pro

In [ ]:
#test_pro.to_csv('submission.csv',index=False)

In [ ]:
test_sq = lgb_train(train2[train2['matchType']==0],test2[test2['matchType']==0]) # 103??
print('test sq complete')

In [ ]:
test_du = lgb_train(train2[train2['matchType']==1],test2[test2['matchType']==1]) # 58??
print('test du complete')

In [ ]:
test_so = lgb_train(train2[train2['matchType']==2],test2[test2['matchType']==2]) # 31??
print('test so complete')

In [ ]:
test_win = pd.concat([test_sq,test_du,test_so])

In [ ]:
sub = pd.read_csv('/kaggle/input/pubg-finish-placement-prediction/sample_submission_V2.csv')

In [ ]:
sub = pd.merge(sub,test_win,on='Id',how='left')

In [ ]:
sub['winPlacePerc'] = sub['winPlacePerc_y']
sub = sub[['Id','winPlacePerc']]

In [ ]:
sub = post_processing(test2,sub['winPlacePerc'])

In [ ]:
sub.to_csv('submission.csv', index=False)

In [ ]:
#y = train2['winPlacePerc']
#X = train2.drop(['winPlacePerc'], axis = 1)

In [ ]:
"""
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2, random_state=42)

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

lightgbm = LGBMRegressor(random_state=42,n_estimators=5000,learning_rate = 0.05,early_stopping_round=20) #num_leaves = 6000
lightgbm.fit(X_train,y_train,eval_set=[(X_val,y_val)], eval_metric='l1')
pred = lightgbm.predict(X_val)
mean_absolute_error(y_val, pred)
"""

In [ ]:
#import seaborn as sns
#importances_values = lightgbm.feature_importances_
#df = pd.DataFrame(importances_values,columns=['importance'],index = X_train.columns)
#df = df.sort_values('importance', ascending=False)
#sns.barplot(data=df, x=df['importance'], y=df.index)

In [ ]:
"""
sub = pd.read_csv('/kaggle/input/pubg-finish-placement-prediction/sample_submission_V2.csv')
pred = lightgbm.predict(test2)
sub['winPlacePerc'] = pred
sub['winPlacePerc'] = sub['winPlacePerc'].apply(lambda x: x if x<=1 else 1)
sub['winPlacePerc'] = sub['winPlacePerc'].apply(lambda x: x if x>=0 else 0)
#sub
"""

In [ ]:
#sub.to_csv('submission.csv', index=False)

In [ ]:
"""
from sklearn.ensemble import AdaBoostRegressor, BaggingRegressor, ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

ada = AdaBoostRegressor(random_state=42)
bag = BaggingRegressor(random_state=42)
extra = ExtraTreesRegressor(random_state=42)
grad = GradientBoostingRegressor(random_state=42)
ran = RandomForestRegressor(random_state=42)
xgb = XGBRegressor(random_state=42)
cat = CatBoostRegressor(random_state=42)
lightgbm = LGBMRegressor(random_state=42, boost_from_average=False) """

In [ ]:
"""score_dic = {}
for col in tqdm(X.columns):
    X_train1 = X_train.drop([col],axis=1)
    X_val1 = X_val.drop([col],axis=1)
    print('delete col:',col)
    lightgbm = LGBMRegressor(random_state=42)
    lightgbm.fit(X_train1,y_train,eval_set=[(X_val1,y_val)], eval_metric='l1',categorical_feature= ['matchType'])
    pred = lightgbm.predict(X_val1)
    score_dic[col] = lightgbm.score(X_val1,y_val)
    print('='*10) 
    
    
    
col = ['damageDealt','headshotPerc','heals','killPoints','vehicleDestroys','healsperdist','boostsperdist','buffperdist','damagePerc']
for col in tqdm(col):
    X_train1 = X_train.drop(col,axis=1)
    X_val1 = X_val.drop(col,axis=1)
    lightgbm = LGBMRegressor(random_state=42,n_estimators=200, learning_rate=0.1)
    lightgbm.fit(X_train1,y_train)
    pred = lightgbm.predict(X_val1)
    print('del:',col,', Mae:',mean_absolute_error(y_val,pred))    
"""

In [ ]:
#lightgbm = LGBMRegressor(random_state=42)
#lightgbm.fit(X_train,y_train,eval_set=[(X_val,y_val)], eval_metric='l1')
#lightgbm.score(X_val,y_val)

In [ ]:
#X_train = X_train[['boosts','killPlace','kills','matchDuration','matchType','numGroups','rideDistance','walkDistance']]
#X_val = X_val[['boosts','killPlace','kills','matchDuration','matchType','numGroups','rideDistance','walkDistance']]

In [ ]:
#test1 = test1[['boosts','killPlace','kills','matchDuration','matchType','numGroups','rideDistance','walkDistance']]

In [ ]:
#X = X.drop(['matchDuration','killPoints','rankPoints','longestKill','teamKills','headshotKills'],axis=1)

#from sklearn.model_selection import train_test_split

#X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2, random_state=42)

In [ ]:
#test1 = test1.drop(['matchDuration','killPoints','rankPoints','longestKill','teamKills','headshotKills'],axis=1)

In [ ]:
"""light_train['buff'] = light_train['heals']+ light_train['boosts']
light_train = light_train.drop(['heals','boosts'],axis=1)

light_val['buff'] = light_val['heals']+ light_val['boosts']
light_val = light_val.drop(['heals','boosts'],axis=1)"""


In [ ]:
"""from sklearn.model_selection import GridSearchCV

lightgbm = LGBMRegressor(random_state=42) #max_depth=17,num_leaves=6000) # estimator 1000  learn 0.3

param_lgb = {"learning_rate" : [0.01,0.1,0.2,0.3,0.4,0.5],
             #"max_depth" : range(10,20),
             #"num_leaves" : [100,300,500,900,1200,4000,6000],
             #"n_estimators" : [100, 200, 300,500,800,1000],
              }

gscv = GridSearchCV (estimator = lightgbm, param_grid = param_lgb, scoring ='neg_mean_absolute_error', cv = 3, refit=True, n_jobs=-1, verbose=2)

gscv.fit(light_train,y_train)

print(gscv.best_params_, gscv.best_score_) """

In [ ]:
#from lightgbm import LGBMRegressor
#lightgbm = LGBMRegressor(random_state=42,n_estimators=200,learning_rate = 0.1,early_stopping_round=20) #num_leaves = 6000
#lightgbm.fit(light_train,y_train,eval_set=[(light_val,y_val)], eval_metric='l1')
#lightgbm.score(light_val, y_val)

In [ ]:
#import seaborn as sns
#importances_values = lightgbm.feature_importances_
#df = pd.DataFrame(importances_values,columns=['importance'],index = light_train.columns)
#df = df.sort_values('importance', ascending=False)
#sns.barplot(data=df, x=df['importance'], y=df.index)

In [ ]:
#xgb.fit(X_train,y_train,eval_set=[(X_val, y_val)],eval_metric='mae')
#print(xgb.score(X_val, y_val))

In [ ]:
#feature_importance =xgb.feature_importances_
#df = pd.DataFrame(feature_importance,columns=['importance'],index = X_train.columns)
#df = df.sort_values('importance', ascending=False)
#sns.barplot(data=df, x=df['importance'], y=df.index)

In [ ]:
#score_dic

In [ ]:
#grad.fit(train1,target)
#print(grad.score(train1, target))

In [ ]:
#lightgbm.fit(train1,target)
#print(lightgbm.score(train1,target))

In [ ]:
#bag.fit(train1,target)
#print(bag.score(train1, target))

In [ ]:
#xgb.fit(train1,target)
#print(xgb.score(train1, target))

In [ ]:
#cat.fit(train1,target,cat_features=['matchType'])
#print(cat.score(train1, target))

In [ ]:
#!pip install pycaret

In [ ]:
#from pycaret.regression import *

In [ ]:
#pyc = setup(data=train1, target='winPlacePerc', train_size=0.8 ,fold=3)

In [ ]:
#best = pyc.compare_models(sort='mae',n_select=3,exclude=['knn','ada','dt','rf','et'])

In [ ]:
#best[1]

In [ ]:
#sub = pd.read_csv('/kaggle/input/pubg-finish-placement-prediction/sample_submission_V2.csv')
#pred = lightgbm.predict(test2)
#sub['winPlacePerc'] = pred
#sub['winPlacePerc'] = sub['winPlacePerc'].apply(lambda x: x if x<=1 else 1)
#sub['winPlacePerc'] = sub['winPlacePerc'].apply(lambda x: x if x>=0 else 0)

In [ ]:
#sub.to_csv('submission.csv', index=False)

In [ ]:
#sub

In [ ]:
#sub[sub['winPlacePerc'] > 1.0]['winPlacePerc']

catboost: 0.92

bagging: 0.98

xgboost: 0.91

adaboost: 0.82

gradinet: 0.89

lightgbm: 0.91